In [ ]:
from src.base_constants import (
    TEST_OUTPUT_PATH,
    DEFAULT_MAX_ITERATIONS,
    DFAULT_CONVERGENCE_THRESHOLD,
)
from src.station import get_stations
from src.simulation_parameters import load_simulation_parameters
from numpy import array, ndarray, matmul, concatenate, mean
from src.utils import save_base_model
from src.quadrature import propagate_partials_and_save, save_normal_equations
from src.test import (
    test_clear_test_folder,
    test_generate_simulation_parameters,
    test_generate_stations,
    test_observations,
    TEST_ARC_PARAMETERS,
)
from numpy.linalg import cholesky, inv, matmul

test_clear_test_folder()
test_generate_simulation_parameters()
test_generate_stations()
test_observations()

output_path = TEST_OUTPUT_PATH
station_file_name = "stations"
arc_id = TEST_ARC_PARAMETERS.arc_id
simulation_parameters_file_name = "simulation_parameters"
parameters_to_invert = [r"J_2"]

stations = get_stations(stations_path=output_path, station_file_name=station_file_name)
simulation_parameters = load_simulation_parameters(
    output_path=output_path,
    arc_id=arc_id,
    name=simulation_parameters_file_name,
)
simulation_parameters.terminal_parameter_values[r"J_2"] = 9e-3
simulation_parameters.update_for_stations(stations=stations)
simulation_parameters.arc_parameters.is_initial = False
observation_partials: dict[str, ndarray[float]]

for iteration in range(DEFAULT_MAX_ITERATIONS):

    arc_output, observation_partials, parameters_to_invert = propagate_partials_and_save(
        stations=stations,
        simulation_parameters=simulation_parameters,
        iteration=iteration,
        parameters_to_invert=parameters_to_invert,
    )

    if mean(abs(concatenate(list(arc_output.residuals.values())))) < DFAULT_CONVERGENCE_THRESHOLD:

        break

    a_matrix = array(object=list(observation_partials.values())).T
    b_second_member = concatenate(list(arc_output.residuals.values()))[:, None]
    n_matrix = array(object=matmul(a_matrix.T, a_matrix), dtype=float)
    s_second_member = array(object=matmul(a_matrix.T, b_second_member), dtype=float)
    path = save_normal_equations(
        n_matrix=n_matrix,
        s_second_member=s_second_member,
        simulation_parameters=simulation_parameters,
        iteration=iteration,
    )
    save_base_model(obj=parameters_to_invert, name="parameters", path=path)
    l_matrix = cholesky(a_matrix)
    z_second_member = matmul(inv(a=l_matrix), s_second_member)
    delta_x_solution = matmul(inv(a=l_matrix.T), z_second_member)

    for delta_x, parameter in zip(delta_x_solution, parameters_to_invert):

        print(parameter, delta_x)
        if parameter in simulation_parameters.terminal_parameter_values:

            simulation_parameters.terminal_parameter_values[parameter] += delta_x